In [ ]:
import pulp as plp

## Beer Transportation Problem

In [ ]:
# Constants and variables

factories = ["A", "B"]
bars = ["1", "2", "3", "4", "5"]
routes = [(f, b) for f in factories for b in bars]

# Maximum supply from each factory
supply = {"A": 1000, "B": 4000}

# Minimum demand from each bar
demand = {"1": 500, "2": 900, "3": 1800, "4": 200, "5": 700}

# Transportation costs
transport_cost = {
    "A": {"1": 2, "2": 4, "3": 5, "4": 2, "5": 1},
    "B": {"1": 3, "2": 1, "3": 3, "4": 2, "5": 3},
}

In [ ]:
problem = plp.LpProblem("beer_transport", sense=plp.LpMinimize)

# Decision variables
crates = plp.LpVariable.dicts(
    "crates", (factories, bars), lowBound=0, upBound=None, cat=plp.LpInteger
)

# Objective function
problem += plp.lpSum([crates[f][b] * transport_cost[f][b] for (f, b) in routes])

# Supply constraint
for f in factories:
    problem += (
        plp.lpSum([crates[f][b] for b in bars]) <= supply[f],
        f"supply_warehouse_{f}",
    )

# Demand constraints
for b in bars:
    problem += (
        plp.lpSum([crates[f][b] for f in factories]) >= demand[b],
        f"demand_bar_{b}",
    )

In [ ]:
# Write problem to file and solve
problem.writeLP("beer_transport.lp")

problem.solve(solver=plp.PULP_CBC_CMD(msg=False))

print("Status:", plp.LpStatus[problem.status])
print("--------")

for route in routes:
    f, b = route
    print(f"Number of crates for route {route} => {crates[f][b].value()}")
print("--------")

for f in factories:
    crates_from_f = sum(crates[f][b].value() for b in bars)
    print(f"Maximum supply for factory {f} = {supply[f]}")
    print(f"Total number of crates from facotry {f} = {crates_from_f}")
print("--------")

for b in bars:
    crates_to_b = sum(crates[f][b].value() for f in factories)
    print(f"Demand at bar {b} = {demand[b]}")
    print(f"Total number of crates to bar {b} = {crates_to_b}")
print("--------")